In [1]:

# 1. Импорт библиотек

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import VarianceThreshold



# 2. Загрузка данных
sheet_id = "1q-nbWuFrfrIBMXmZfNW78N3bx5v60Vb9"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

df = pd.read_csv(url)



# 3. Очистка данных

df = df.drop(columns=['Unnamed: 0'])
df = df.fillna(df.median(numeric_only=True))

selector = VarianceThreshold(threshold=0.01)
X_temp = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])

selector.fit(X_temp)

low_var_cols = X_temp.columns[~selector.get_support()]
df = df.drop(columns=low_var_cols)



# 4. Подготовка данных

X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])
y = df['CC50, mM']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# 5. Linear Regression

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print("Linear Regression")
print("MAE:", mean_absolute_error(y_test, y_pred_lr))
print("MSE:", mean_squared_error(y_test, y_pred_lr))
print("R2:", r2_score(y_test, y_pred_lr))



# 6. Decision Tree

tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)

y_pred_tree = tree.predict(X_test)

print("\nDecision Tree")
print("MAE:", mean_absolute_error(y_test, y_pred_tree))
print("MSE:", mean_squared_error(y_test, y_pred_tree))
print("R2:", r2_score(y_test, y_pred_tree))



# 7. Random Forest

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("\nRandom Forest")
print("MAE:", mean_absolute_error(y_test, y_pred_rf))
print("MSE:", mean_squared_error(y_test, y_pred_rf))
print("R2:", r2_score(y_test, y_pred_rf))

Linear Regression
MAE: 372.14523043574343
MSE: 321922.9842259173
R2: 0.3790666802871989

Decision Tree
MAE: 343.1521587783897
MSE: 368781.527780761
R2: 0.28868471804123164

Random Forest
MAE: 287.2105141605055
MSE: 210820.0886376894
R2: 0.5933647986809121


### Что получилось

**Linear Regression**
- MAE: **372.1**
- MSE: **321923.0**
- R2: **0.379**

**Decision Tree**
- MAE: **343.2**
- MSE: **368781.5**
- R2: **0.289**

**Random Forest**
- MAE: **287.2**
- MSE: **210820.1**
- R2: **0.593**

### Вывод

- Линейная модель даёт средний baseline  
- Decision Tree отработала хуже всех  
- Лучший результат у **Random Forest**  

 для CC50 тоже основной моделью пока становится **Random Forest**

Следующее:
- делаю облегчённый подбор параметров для Random Forest

In [2]:
# Облегчённый подбор параметров для CC50
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

random_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Лучшие параметры:", random_search.best_params_)

best_rf = random_search.best_estimator_
y_pred = best_rf.predict(X_test)

print("\nПосле подбора:")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

Лучшие параметры: {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 10}

После подбора:
MAE: 288.2099014349911
MSE: 210537.87380075364
R2: 0.5939091419063331


### Что делаю

- Подбираю параметры Random Forest через `RandomizedSearchCV`
- Перебираю:
   `n_estimators`
   `max_depth`
   `min_samples_split`
   `min_samples_leaf`

### Что получилось

Лучшие параметры:
- `n_estimators = 100`
- `max_depth = 10`
- `min_samples_split = 5`
- `min_samples_leaf = 1`

После подбора:
- **MAE: 288.2**
- **MSE: 210537.9**
- **R2: 0.593**

### Вывод

- Качество практически не изменилось (R2 осталось ~0.593)
- Подбор параметров не дал улучшения  

 оставляю базовый Random Forest как финальную модель для CC50

### Итоговый вывод по CC50

- Лучшая модель: **Random Forest**
- Качество:
   **R2 ≈ 0.59**
   ошибка ниже, чем у остальных моделей  

- Linear Regression даёт средний результат  
- Decision Tree работает хуже  
- Подбор гиперпараметров не улучшил модель  

→ фиксирую Random Forest как итоговую модель  

Следующее:
- создаю файл **model_si.ipynb**